# LEC 5 / 6 ML Basics

## CONCEPT — The 6-Step ML Workflow

Step 1: Load Data → pandas read_csv()

Step 2: Explore Data → df.head(), df.describe(), df.info()

Step 3: Split Features/Labels → X = features, y = target

Step 4: Train-Test Split → 80% train, 20% test

Step 5: Choose & Train Model → model.fit(X_train, y_train)

Step 6: Evaluate → accuracy, R², MSE, confusion matrix


Split Ratio

90/10	Very small datasets (<500 rows)	Too few test samples — unreliable evaluation

80/20	Most common — balanced choice	Safe default for most problems

75/25	Larger datasets (>10K rows)	More reliable evaluation

60/20/20	With validation set too	Train/Validate/Test — used in hyperparameter tuning



## Linear Regression — Predicting a Continuous Number

In [ ]:
# Linear Regression model

#load and prepare

import pandas as pd

df = pd.read_csv('grade_set1.csv')

X = df[['hours_studied']]
y = df['test_grade']


# train and test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# training linear regression

from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train) #model training happens here

print('slope: ', lr.coef_, 'Intercept: ', lr.intercept_)  # coefficeint is slope as in how much y changed by each unit of x, how much grade changes per extra hour of study, intercept is bias, how much one can score even without studying


# making prediction

y_pred = lr.predict(X_test)   # get prediction on test for scores

df['test_grade_predict'] = lr.predict(X) # predicts  score of full dataset, this is just for analysis

# model evaluation / loss function

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print('R- Squared: ', r2_score(y_test, y_pred))    # proportion of variance explained. R²=0.97 means 97% of variability is explained by the model
print('MAE: ', mean_absolute_error(y_test,y_pred))  # average of |actual - predicted|. More robust to outliers.
print('RMSE: ', np.sqrt(mean_squared_error(y_test, y_pred)))   # MSE - average of (actual - predicted)². Penalises big errors heavily.rmse  = √MSE easier to interpret. 'Off by X grade points on average.'


# visualising the regression line

# plot actual VS predicted

import matplotlib.pyplot as plt

plt.scatter(X, y , color = 'blue', lable = 'Actual')  # create dots for actul data point hours studied and marks
plt.plot(X, lr.predict(X), color = 'red', lable = 'Predicted') # creates line for predicted marks for actualy hours studied
plt.legend()
plt.show()



##Logistic Regression — Classification Despite the Name

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression() # get the model
clf.fit(X_train, y_train)  # train the model
y_pred = clf.predict(X_test)  # predict the y for testing set , Returns class labels (0 or 1). Use clf.predict_proba(X_test) if you want the raw probabilities.

# model evaluation / loss function
# Getting Probabilities and Classification Report
# log loss (binary cross entropy)

proba = clf.predict_proba(X_test)[:,1]

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred)) # Prints precision, recall, F1 for EACH class. The fastest way to diagnose your classifier — always run this.



# LEC 5/6 ML Part 2

## Decision Tree

In [ ]:
# training decision tree classifier

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth = 5 , criterion = 'gini')
dt.fit(X_train, y_train)
print(dt.score(X_test, y_test))

# model evaluation with Accuracy, Confusion Matrix, Classification Report

from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.metrics import classification_report
y_pred = dt.predict(X_test)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))



## Random Forest

In [ ]:
# training random forests

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators = 100, max_depth = 5,
                            oob_score= True, random_state = 42 )

rf.fit(X_train, y_train)

#getting oob score
print("OOB Score: ", rf.oob_score_)

# feature importance

import pandas as pd

importances = pd.Series(rf.feature_importances_, index=feature_names)

importances.sort_values().plot(kind='barh')




In [ ]:
# Creating Polynomial Features
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

lr = LinearRegression().fit(X_poly, y)

# Comparing Underfitting vs Overfitting Visually
import matplotlib.pyplot as plt
import numpy as np
degrees = [1, 4, 15]   # underfit, good, overfit
for d in degrees:
    poly = PolynomialFeatures(degree=d)
    plt.plot(X_line, lr.predict(poly.transform(X_line)))



## Gradient Descent

Gradient descent says: "move weights downhill to reduce loss."

Optimiser says: "here's exactly HOW we move — fast, slow, smart, simple."


You're blindfolded on a hill, trying to reach the lowest point.

SGD:       take one random step downhill each time → fast but wobbly

Mini-batch: average a few steps then move → more stable

Adam:      checks how steep the hill is AND remembers past steps → smartest path

RMSprop: similar to Adam but without momentum
historically used for RNNs (text/sequence data)
Adam has mostly replaced it now


In [ ]:
#gradient descent
# Comparing Optimisers in PyTorch
import torch.optim as optim
sgd   = optim.SGD(model.parameters(), lr=0.01)
adam  = optim.Adam(model.parameters(), lr=0.001)
rmsp  = optim.RMSprop(model.parameters(), lr=0.001)
